# Issue #4 — Baseline e comparação de redes Keras

**Objetivo:** comparar regressão logística e três redes neurais pelo F1 da classe `Churn=Yes`, usando **a mesma divisão estratificada 60/20/20 da issue #3** (seed 42).

O **teste (20%) é cofre**: este notebook não avalia o conjunto de teste. A escolha de arquitetura e limiar usa exclusivamente a validação (20%).

**Para executar:** instale `requirements.txt`, mantenha este notebook na raiz do repositório e execute as células em ordem. O CSV será baixado com `download_csv.py` caso ainda não exista (requer acesso à internet).

In [ ]:
from pathlib import Path
import json
import random
import subprocess
import sys
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report, ConfusionMatrixDisplay
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

ARTEFATOS = Path('artefatos')
ARTEFATOS.mkdir(exist_ok=True)
print('TensorFlow:', tf.__version__)

## 1. Reutilizar o protocolo da issue #3

Mesmas duas chamadas a `train_test_split`, mesmos nomes de features e tratamento de `TotalCharges`, `StandardScaler` e `OneHotEncoder`. O pré-processador é ajustado **somente** no treino.

In [ ]:
CAMINHO_CSV = Path('WA_Fn-UseC_-Telco-Customer-Churn.csv')
if not CAMINHO_CSV.exists():
    print('CSV ausente. Executando download_csv.py ...')
    subprocess.run([sys.executable, 'download_csv.py'], check=True)
assert CAMINHO_CSV.exists(), 'CSV não encontrado. Coloque o dataset na raiz do projeto.'

df = pd.read_csv(CAMINHO_CSV)
df_resto, df_teste = train_test_split(
    df, test_size=0.20, random_state=SEED, stratify=df['Churn']
)
df_treino, df_valid = train_test_split(
    df_resto, test_size=0.25, random_state=SEED, stratify=df_resto['Churn']
)

def limpar_dados(dados):
    d = dados.copy()
    d['TotalCharges'] = pd.to_numeric(d['TotalCharges'], errors='coerce').fillna(0.0)
    d['Churn_num'] = d['Churn'].map({'No': 0, 'Yes': 1})
    return d

df_treino = limpar_dados(df_treino)
df_valid = limpar_dados(df_valid)
# df_teste permanece reservado, sem processamento nem métricas nesta issue.

FEATURES_EXCLUIDAS = ['customerID', 'Churn', 'Churn_num']
X_treino = df_treino.drop(columns=FEATURES_EXCLUIDAS)
y_treino = df_treino['Churn_num'].to_numpy(dtype=np.int32)
X_valid = df_valid.drop(columns=FEATURES_EXCLUIDAS)
y_valid = df_valid['Churn_num'].to_numpy(dtype=np.int32)

colunas_numericas = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
colunas_categoricas = [c for c in X_treino.columns if c not in colunas_numericas]
preprocessador = ColumnTransformer(transformers=[
    ('num', StandardScaler(), colunas_numericas),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), colunas_categoricas),
])
X_treino_proc = np.asarray(preprocessador.fit_transform(X_treino), dtype=np.float32)
X_valid_proc = np.asarray(preprocessador.transform(X_valid), dtype=np.float32)

print(f'Treino: {len(y_treino)} | Validação: {len(y_valid)} | Teste reservado: {len(df_teste)}')
print('Dimensão após preprocessamento:', X_treino_proc.shape[1])
print(f'Churn=Yes — treino: {y_treino.mean():.3f} | validação: {y_valid.mean():.3f}')

## 2. Métrica e seleção de limiar

A probabilidade é convertida em classe positiva se `p >= limiar`. Testamos limiares pré-definidos entre 0,10 e 0,90 na **validação**; o desempate favorece o limiar mais próximo de 0,50. Evitamos usar o teste para escolher o limiar.

In [ ]:
LIMIARES = np.round(np.arange(0.10, 0.901, 0.025), 3)

def escolher_limiar(y_real, probabilidades):
    candidatos = []
    for limiar in LIMIARES:
        previsoes = (probabilidades >= limiar).astype(int)
        candidatos.append({
            'limiar': float(limiar),
            'f1_yes': float(f1_score(y_real, previsoes, pos_label=1, zero_division=0)),
            'precision_yes': float(precision_score(y_real, previsoes, pos_label=1, zero_division=0)),
            'recall_yes': float(recall_score(y_real, previsoes, pos_label=1, zero_division=0)),
        })
    return sorted(candidatos, key=lambda r: (-r['f1_yes'], abs(r['limiar'] - 0.5), r['limiar']))[0]

resultados = []
historicos = {}
modelos_treinados = {}

## 3. Baseline — regressão logística

In [ ]:
baseline = LogisticRegression(max_iter=2000, random_state=SEED)
baseline.fit(X_treino_proc, y_treino)
prob_baseline = baseline.predict_proba(X_valid_proc)[:, 1]
metricas_baseline = escolher_limiar(y_valid, prob_baseline)
resultados.append({
    'modelo': 'RegressaoLogistica', 'tipo': 'baseline',
    'arquitetura': 'LogisticRegression(max_iter=2000)',
    'dropout': 0.0, 'l2': 0.0, 'learning_rate': None,
    'batch_size': None, 'epocas_executadas': None,
    'melhor_val_loss': None, **metricas_baseline,
})
modelos_treinados['RegressaoLogistica'] = baseline
print('Baseline:', metricas_baseline)

## 4. Três arquiteturas Keras

Todas têm **saída sigmoide**, `binary_crossentropy`, monitoramento de `val_loss`, `EarlyStopping(restore_best_weights=True)` e mesma validação. A variação de camadas, dropout e regularização permite uma comparação real.

In [ ]:
CONFIGURACOES = {
    'Keras_1': {'camadas': [16], 'dropout': 0.0, 'l2': 0.0, 'lr': 0.001, 'batch': 32},
    'Keras_2': {'camadas': [64, 32], 'dropout': 0.20, 'l2': 0.0, 'lr': 0.001, 'batch': 32},
    'Keras_3': {'camadas': [128, 64, 32], 'dropout': 0.30, 'l2': 0.001, 'lr': 0.0005, 'batch': 64},
}

def construir_rede(config, num_features):
    modelo = keras.Sequential(name='rede_churn')
    modelo.add(keras.Input(shape=(num_features,)))
    for unidades in config['camadas']:
        modelo.add(layers.Dense(
            unidades, activation='relu',
            kernel_regularizer=regularizers.l2(config['l2'])
        ))
        if config['dropout'] > 0:
            modelo.add(layers.Dropout(config['dropout']))
    modelo.add(layers.Dense(1, activation='sigmoid'))
    modelo.compile(
        optimizer=keras.optimizers.Adam(learning_rate=config['lr']),
        loss='binary_crossentropy',
        metrics=[keras.metrics.BinaryAccuracy(name='accuracy')]
    )
    return modelo

for idx, (nome, config) in enumerate(CONFIGURACOES.items()):
    print(f'\nTreinando {nome}: {config}')
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED + idx)
    modelo = construir_rede(config, X_treino_proc.shape[1])
    historico = modelo.fit(
        X_treino_proc, y_treino,
        validation_data=(X_valid_proc, y_valid),
        epochs=150,
        batch_size=config['batch'],
        callbacks=[keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=12, min_delta=0.0001,
            restore_best_weights=True
        )],
        verbose=0,
    )
    probabilidades = modelo.predict(X_valid_proc, verbose=0).ravel()
    metricas = escolher_limiar(y_valid, probabilidades)
    historicos[nome] = historico.history
    modelos_treinados[nome] = modelo
    resultados.append({
        'modelo': nome, 'tipo': 'keras',
        'arquitetura': '-'.join(map(str, config['camadas'])),
        'dropout': config['dropout'], 'l2': config['l2'],
        'learning_rate': config['lr'], 'batch_size': config['batch'],
        'epocas_executadas': len(historico.history['loss']),
        'melhor_val_loss': min(historico.history['val_loss']),
        **metricas,
    })
    print('Resultado validação:', metricas)
    # Salva cada tentativa, sem depender do modelo vencedor.
    modelo.save(ARTEFATOS / f'{nome}.keras')

## 5. Curvas de treino e validação

Curvas de `loss` e de `accuracy` para as três tentativas. O baseline não tem curva por época, pois não é uma rede treinada por `epochs` no Keras.

In [ ]:
fig, eixos = plt.subplots(3, 2, figsize=(12, 12))
for linha, (nome, hist) in enumerate(historicos.items()):
    for coluna, metrica in enumerate(['loss', 'accuracy']):
        eixos[linha, coluna].plot(hist[metrica], label='Treino')
        eixos[linha, coluna].plot(hist['val_' + metrica], label='Validação')
        eixos[linha, coluna].set_title(f'{nome} — {metrica}')
        eixos[linha, coluna].set_xlabel('Época')
        eixos[linha, coluna].legend()
fig.tight_layout()
fig.savefig(ARTEFATOS / 'curvas_treinamento_validacao.png', dpi=150)
plt.show()

![Curvas de perda e acurácia no treino e na validação das três redes Keras](artefatos/curvas_treinamento_validacao.png)

## 6. Comparação por F1 de `Churn=Yes` e registro

`experimentos.csv` registra **baseline + três tentativas** com arquitetura, hiperparâmetros, época, limiar e métricas de validação.

**Atenção:** o `.gitignore` original ignora `*.csv`; para versionar `experimentos.csv`, será necessário adicionar uma exceção no `.gitignore`.

In [ ]:
experimentos = pd.DataFrame(resultados).sort_values(
    ['f1_yes', 'precision_yes'], ascending=False
).reset_index(drop=True)
display(experimentos)
experimentos.to_csv('experimentos.csv', index=False)

ax = experimentos.sort_values('f1_yes').plot.barh(
    x='modelo', y='f1_yes', legend=False, figsize=(9, 4)
)
ax.set_xlabel('F1 na validação — Churn=Yes')
ax.set_ylabel('Modelo')
plt.tight_layout()
plt.savefig(ARTEFATOS / 'comparacao_f1_validacao.png', dpi=150)
plt.show()

## 7. Escolha definitiva **antes** da avaliação final

Selecionamos o maior F1 em validação, mas exportamos **também a melhor rede Keras**, mesmo se a regressão logística ganhar — a issue solicita disponibilizar uma rede neural para a interface. Os dois resultados são registrados sem confusão. Nenhum acesso ao conjunto de teste ocorre aqui.

In [ ]:
vencedor_geral = experimentos.iloc[0].to_dict()
melhor_keras = experimentos.query("tipo == 'keras'").iloc[0].to_dict()

config_escolhida = {
    'criterio': 'F1 da classe Churn=Yes na validacao; desempate por precision',
    'baseline_ou_rede_vencedor_geral': vencedor_geral['modelo'],
    'f1_vencedor_geral': float(vencedor_geral['f1_yes']),
    'limiar_vencedor_geral': float(vencedor_geral['limiar']),
    'rede_selecionada_para_interface': melhor_keras['modelo'],
    'arquitetura_rede': melhor_keras['arquitetura'],
    'limiar_rede': float(melhor_keras['limiar']),
    'f1_rede_validacao': float(melhor_keras['f1_yes']),
    'features_esperadas': X_treino.columns.tolist(),
    'seed': SEED,
    'proporcao': {'treino': 0.60, 'validacao': 0.20, 'teste_reservado': 0.20},
    'avaliacao_teste_realizada': False,
}

with open(ARTEFATOS / 'configuracao_escolhida.json', 'w', encoding='utf-8') as f:
    json.dump(config_escolhida, f, indent=2, ensure_ascii=False)

joblib.dump(preprocessador, ARTEFATOS / 'preprocessador.joblib')
modelos_treinados[melhor_keras['modelo']].save(ARTEFATOS / 'modelo_churn.keras')
joblib.dump(baseline, ARTEFATOS / 'baseline_logistico.joblib')

print('Escolha registrada ANTES de avaliar o teste:')
print(json.dumps(config_escolhida, indent=2, ensure_ascii=False))
print('Arquivos exportados em:', ARTEFATOS.resolve())

## 8. Verificar integração da rede selecionada

A aplicação deve carregar **o mesmo pré-processador, a mesma sequência de colunas e o limiar registrado**. A célula abaixo faz apenas um teste local com uma linha de **validação**, nunca com teste final.

In [ ]:
preprocessador_carregado = joblib.load(ARTEFATOS / 'preprocessador.joblib')
rede_carregada = keras.models.load_model(ARTEFATOS / 'modelo_churn.keras')
with open(ARTEFATOS / 'configuracao_escolhida.json', encoding='utf-8') as f:
    parametros = json.load(f)

amostra_validacao = X_valid.iloc[[0]][parametros['features_esperadas']]
entrada = np.asarray(preprocessador_carregado.transform(amostra_validacao), dtype=np.float32)
prob = float(rede_carregada.predict(entrada, verbose=0).ravel()[0])
limiar = parametros['limiar_rede']
print({'probabilidade_Churn_Yes': round(prob, 4), 'limiar': limiar, 'classe_prevista': 'Yes' if prob >= limiar else 'No'})

### Checklist da entrega

- `treinamento_modelos.ipynb`: código e curvas de treino/validação.
- `experimentos.csv`: três redes + baseline, F1 e limiar.
- `artefatos/modelo_churn.keras`: rede selecionada e treinada.
- `artefatos/preprocessador.joblib`: pré-processamento compatível.
- `artefatos/configuracao_escolhida.json`: decisão fixada **antes do teste final**.
- `artefatos/curvas_treinamento_validacao.png` e `artefatos/comparacao_f1_validacao.png`.

**Importante:** o conjunto de teste deve ser avaliado uma única vez em uma etapa posterior, após congelar a escolha. Não reotimize o modelo com base no teste.